#only use this on the jupyterhub first time use

#this is the training part

In [ ]:
from ultralytics import YOLO
import yaml
from datetime import datetime
import shutil
import os
import pathlib
import torch
import gc

In [5]:


with open("settings.yaml", "r") as f:
    settings = yaml.safe_load(f)

training_datasets = settings["training_datasets"]
training_configurations = settings["training_configurations"]
model_paths = ["yolo11x.pt", "yolo11n.pt"]  

start_time = datetime.now()

for dataset in training_datasets:
    dataset_yaml = pathlib.Path.cwd().parent / "datasets" / dataset["type"] / dataset["name"] / "dataset.yaml"
    for configuration in training_configurations:
        for model_path in model_paths:
            model = YOLO(model_path)
            model_name = pathlib.Path(model.ckpt_path).stem
            timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M")
            run_name = f"{timestamp}_{model_name}"

            train_kwargs = {
                "data": str(dataset_yaml),
                "name": run_name,
                **{k: configuration[k] for k in ["time", "imgsz", "patience", "epochs", "multi_scale", "batch"] if k in configuration}
            }

            train_results = model.train(**train_kwargs)

            # Move results directory
            for subdir in ["detect", "segment"]:
                source_dir = pathlib.Path("runs") / subdir / run_name
                if source_dir.is_dir():
                    target_dir = pathlib.Path.cwd().parent / "models" / dataset["type"] / run_name
                    shutil.move(str(source_dir), str(target_dir))
                    print(f"Model saved to: {target_dir}")
                    break
            else:
                print("No valid run directory found")

            # Print training info
            end_time = datetime.now()
            duration = end_time - start_time
            hours, minutes = divmod(int(duration.total_seconds() // 60), 60)
            print(f"Training of {dataset['name']} with {model_name} completed")
            print(f"Training duration: {hours}h {minutes}m")

            # Release memory
            del train_results
            del model
            torch.cuda.empty_cache()

Ultralytics 8.3.166 🚀 Python-3.10.18 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 2070 with Max-Q Design, 7787MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/freeze/GIT/onemanstreasure/datasets/multiclass/der_merger-400/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11x.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=2025_09_03_17_51_yolo11x, nbs=64, nms=False, opset=None, optimize=Fals

train: Scanning /home/freeze/GIT/onemanstreasure/datasets/multiclass/der_merger-400/labels/train... 294 images, 19 backgrounds, 0 corrupt: 100%|██████████| 294/294 [00:00<00:00, 3172.32it/s]

train: New cache created: /home/freeze/GIT/onemanstreasure/datasets/multiclass/der_merger-400/labels/train.cache
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (NVIDIA GeForce RTX 2070 with Max-Q Design) 7.60G total, 0.56G reserved, 0.54G allocated, 6.50G free


      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output
    56878396       195.5         3.544         47.23         81.31        (1, 3, 640, 640)                    list
    56878396       390.9         4.922         52.57            99        (2, 3, 640, 640)                    list
    56878396       781.9         7.179         103.4         190.3        (4, 3, 640, 640)                    list
CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 7.60 GiB of which 1.75 MiB is free. Process 26504 has 258.00 MiB memory in use. Including non-PyTorch memory, this process has 7.15 GiB memory in use. Of the allocated memory 6.85 GiB is allocated by PyTorch, and 119.00 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/

train: Scanning /home/freeze/GIT/onemanstreasure/datasets/multiclass/der_merger-400/labels/train.cache... 294 images, 19 backgrounds, 0 corrupt: 100%|██████████| 294/294 [00:00<?, ?it/s]


...

Plotting labels to runs/detect/2025_09_03_17_52_yolo11n/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.00125, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.00053125), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs/detect/2025_09_03_17_52_yolo11n
Starting training for 1 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


        1/1      2.33G      1.183      3.585      1.477          8        640: 100%|██████████| 18/18 [00:03<00:00,  5.62it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:00<00:00,  5.11it/s]


                   all         36         56    0.00926      0.802      0.174      0.086

1 epochs completed in 0.002 hours.
Optimizer stripped from runs/detect/2025_09_03_17_52_yolo11n/weights/last.pt, 5.4MB
Optimizer stripped from runs/detect/2025_09_03_17_52_yolo11n/weights/best.pt, 5.4MB

Validating runs/detect/2025_09_03_17_52_yolo11n/weights/best.pt...
Ultralytics 8.3.166 🚀 Python-3.10.18 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 2070 with Max-Q Design, 7787MiB)
YOLO11n summary (fused): 100 layers, 2,582,932 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:00<00:00,  9.42it/s]


                   all         36         56    0.00927      0.802      0.176     0.0855
                 Chair          3          5   0.000989        0.6      0.013     0.0117
                Window         19         36     0.0161      0.806      0.181     0.0889
                  Door         14         15     0.0107          1      0.333      0.156
Speed: 0.2ms preprocess, 2.7ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to runs/detect/2025_09_03_17_52_yolo11n
Model saved to: /home/freeze/GIT/onemanstreasure/models/multiclass/2025_09_03_17_52_yolo11n
Training of der_merger-400 with yolo11n completed
Training duration: 0h 1m


In [3]:
!nvidia-smi

Wed Sep  3 17:42:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.169                Driver Version: 570.169        CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2070 ...    Off |   00000000:01:00.0  On |                  N/A |
| N/A   65C    P0             29W /   80W |     844MiB /   8192MiB |     27%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----